<img align="center" src="https://iili.io/3wI8gI.png" style="height:90px" style="width:30px"/>

<h1><center>Supervised ML Regression Competition</center></h1>
You have been assigned the task of building a model that will predict the insurance cost of a customer


<hr style="border:2px solid pink"> </hr>


<img align="center" src="https://compraracciones.com/wp-content/uploads/2021/04/insurance.jpg" style="height:200px" style="width:100px"/>

<hr style="border:2px solid pink"> </hr>

You'll find the data in the csv file `insurance`


- target col: "charges"



**Guidelines:** 


- train_test_split
    - random state = 42
    - test size = 0.3



In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,7725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,15799.570247
std,14.049960,6.098187,1.205493,12559.984021
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,8106.480000
50%,39.000000,30.400000,1.000000,12403.585225
75%,51.000000,34.693750,2.000000,18948.612903
max,64.000000,53.130000,5.000000,64571.074480


In [4]:
X = df.drop("charges", axis=1)
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

numeric_cols = ["age", "bmi", "children"]
categorical_cols = ["sex", "smoker", "region"]

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_cols),
    (OneHotEncoder(drop="if_binary"), categorical_cols),
)

## Baseline: Linear Regression

In [5]:
lin_pipe = make_pipeline(preprocessor, LinearRegression())
lin_pipe.fit(X_train, y_train)

y_pred = lin_pipe.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.4f}")

RMSE: 6558.51
MAE: 4996.26
R2: 0.7263


## GridSearchCV: Ridge Regression

In [6]:
ridge_pipe = make_pipeline(preprocessor, Ridge())

param_grid = {
    "ridge__alpha": [0.01, 0.1, 1, 10, 50, 100, 200]
}

grid = GridSearchCV(ridge_pipe, param_grid=param_grid, scoring="r2", cv=5)
grid.fit(X_train, y_train)

print(f"Best score: {grid.best_score_}\nBest params: {grid.best_params_}")

Best score: 0.6821497591612197
Best params: {'ridge__alpha': 1}


In [7]:
best_ridge = grid.best_estimator_
y_pred = best_ridge.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.4f}")

RMSE: 6561.87
MAE: 5003.94
R2: 0.7261


## GridSearchCV: Random Forest Regressor

In [8]:
rf_pipe = make_pipeline(preprocessor, RandomForestRegressor(random_state=42))

param_grid = {
    "randomforestregressor__n_estimators": [100, 300, 500],
    "randomforestregressor__max_depth": [3, 5, 8, None],
}

rf_grid = GridSearchCV(rf_pipe, param_grid=param_grid, scoring="r2", cv=5)
rf_grid.fit(X_train, y_train)

print(f"Best score: {rf_grid.best_score_}\nBest params: {rf_grid.best_params_}")

Best score: 0.8465462766293012
Best params: {'randomforestregressor__max_depth': 5, 'randomforestregressor__n_estimators': 100}


In [9]:
best_rf = rf_grid.best_estimator_
y_pred = best_rf.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.4f}")

RMSE: 4401.22
MAE: 2582.01
R2: 0.8768
